# MS_C3 — Multi-Station GNN-LSTM + Stacking (Perfect Forecast Full: HYSPLIT + MET at t+h)

Trains **one shared multi-output GNN-LSTM** over all stations, then a per-station
tree-stacking ensemble. Uses `weather_mode="perfect_forecast_full"` throughout (GNN node
features and tabular stacking base models both receive MET_COLS + HYSPLIT — numeric **and**
direction one-hots — shifted to t+24).

**Phase A — shared GNN (trained once):** all stations are nodes *and* readout targets. The
graph convolution shares weights across nodes, so a single training produces a forecast for
every station: `predict_gnn_lstm` returns `(n_samples, n_nodes, n_horizons)`. Forecasts for
station *k* are read out from node *k*. Trained once; reloaded from a shared checkpoint if present.

**Phase B — per-station stacking:** for each station, slice the GNN's node-*k* predictions
and feed them (as before) into the tree-stacking ensemble. Writes the same per-station
`C3_metrics.csv` / `C3_GNN_metrics.csv` / `C3_Stacking_metrics.csv` schema as previous runs.

**Checkpoint:** Phase B skips a station if `outputs/{station}/results/C3_metrics.csv` already
exists. The shared GNN trains up front (and reloads from its checkpoint on a kernel restart),
so a restart resumes from the last incomplete station's stacking.

> Note: the GNN's node features intentionally exclude flat neighbor-PM lags — the graph
> convolution aggregates neighbor PM2.5 across nodes, so adding them would double-count.
> Dynamic (wind/trajectory-informed) adjacency is implemented in `dynamic_adjacency()` but is
> **not** wired in here; this notebook trains on the static distance-based adjacency.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe
import src.models.gnn_stacking as gs

from src.config import (
    ALL_STATIONS, NEIGHBOR_STATIONS, HORIZONS, RANDOM_SEED,
    STATION_COORDS, get_station_paths,
)
from src.feature_engineering import build_feature_matrix, WEATHER_MODE_PERFECT_FULL
from src.models.gnn_stacking import (
    build_adjacency_matrix, build_gnn_sequence_dataset,
    train_gnn_lstm, predict_gnn_lstm,
    train_stacking_ensemble, predict_stacking,
)
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = WEATHER_MODE_PERFECT_FULL
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE}')

Stations: ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']
Weather mode: perfect_forecast_full


In [2]:
# Stacking base model factories (same as B3 single-station notebook)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

def make_base_models():
    return {
        'xgb': lambda: TransformedTargetRegressor(
            regressor=XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                   random_state=RANDOM_SEED, verbosity=0, tree_method='hist'),
            func=np.log1p, inverse_func=np.expm1),
        'hgb': lambda: TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(
                max_iter=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_SEED),
            func=np.log1p, inverse_func=np.expm1),
        'lgb': lambda: TransformedTargetRegressor(
            regressor=lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                        random_state=RANDOM_SEED, verbose=-1),
            func=np.log1p, inverse_func=np.expm1),
        'cat': lambda: CatBoostRegressor(iterations=300, learning_rate=0.05, depth=4,
                                         random_seed=RANDOM_SEED, verbose=0, loss_function='MAE'),
    }

In [3]:
# ════════════════════════════════════════════════════════════════════
# PHASE A — train ONE shared multi-output GNN-LSTM for all stations
# ════════════════════════════════════════════════════════════════════
SEQ_LEN_GNN = 48
META_FEAT_COLS_BASE = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'blh', 'pm25_now']

wall_start = time.time()

# Node order is the fixed canonical station list (target-independent now).
gs.ALL_STATIONS_ORDERED = list(ALL_STATIONS)
ALL_NODES = gs.ALL_STATIONS_ORDERED
print(f'GNN node order: {ALL_NODES}')

# Shared GNN reads the full station table once; no per-station TARGET monkeypatching.
df = dl.load_data()
train_df, test_df = dl.train_test_split(df)

# ── Build GNN sequence datasets (per-node targets) ──────────────────
X_gnn_train, y_gnn_train, A_static, gnn_scaler = build_gnn_sequence_dataset(
    train_df, seq_len=SEQ_LEN_GNN, horizons=HORIZONS, weather_mode=WEATHER_MODE,
    fit_scaler=True
)
X_gnn_test, y_gnn_test, _, _ = build_gnn_sequence_dataset(
    test_df, seq_len=SEQ_LEN_GNN, horizons=HORIZONS, weather_mode=WEATHER_MODE,
    scaler=gnn_scaler
)
print(f'GNN Train: {X_gnn_train.shape} | Test: {X_gnn_test.shape}')  # (n, seq, n_nodes, F)

# ── Validation split: fixed mid-training block (2022) ────────────────
# Identical strategy to MS_C2 — anchored to start of 2022 so early stopping
# is not tuned toward the regime immediately before the 2024 test set.
val_start = int(3 * 8760)
val_end   = val_start + int(0.1 * len(X_gnn_train))
val_end   = min(val_end, len(X_gnn_train) - 1)
X_gnn_tr  = np.concatenate([X_gnn_train[:val_start], X_gnn_train[val_end:]], axis=0)
y_gnn_tr  = np.concatenate([y_gnn_train[:val_start], y_gnn_train[val_end:]], axis=0)
X_gnn_val = X_gnn_train[val_start:val_end]
y_gnn_val = y_gnn_train[val_start:val_end]

# ── Train (or reload) the single shared GNN-LSTM ─────────────────────
# Shared checkpoint lives under the first station's models dir; reused by all.
shared_models_dir = get_station_paths(ALL_NODES[0])['models']
ensure_dirs(shared_models_dir)
gnn_model_path = shared_models_dir / 'gnn_lstm_shared_pfxf_model.pt'

import torch
if gnn_model_path.exists():
    print(f'Reloading shared GNN checkpoint → {gnn_model_path}')
    ckpt = torch.load(gnn_model_path, map_location='cpu', weights_only=False)
    gnn_model = gs.build_gnn_lstm_model(
        ckpt['n_nodes'], ckpt['n_node_features'], ckpt['seq_len'], ckpt['n_horizons'])
    gnn_model.load_state_dict(ckpt['model_state'])
    gnn_model.eval()
    assert ckpt['all_stations'] == ALL_NODES, 'Checkpoint node order differs from current ALL_STATIONS'
else:
    gnn_model, _ = train_gnn_lstm(
        X_gnn_tr, y_gnn_tr, X_gnn_val, y_gnn_val, A_static,
        epochs=100, batch_size=64, patience=15, lr=1e-3,
        save_path=gnn_model_path, scaler=gnn_scaler
    )

# Predict once for ALL stations: (n_samples, n_nodes, n_horizons)
gnn_train_preds = predict_gnn_lstm(gnn_model, X_gnn_train, A_static)
gnn_test_preds  = predict_gnn_lstm(gnn_model, X_gnn_test,  A_static)
# Per-node ground truth (test) in µg/m³ for GNN evaluation in Phase B.
y_gnn_test_true = np.expm1(y_gnn_test)  # (n_test, n_nodes, n_horizons)

print(f'Shared GNN predictions | train {gnn_train_preds.shape} | test {gnn_test_preds.shape}')
print(f'Phase A complete in {(time.time() - wall_start) / 60:.1f}min.')

09:09:41 | src.data_loader | INFO | Loading data from D:\MOJE\PROJEKTY\warsaw_aq_forecast\data\raw\FINAL_merged_PM25_1g_all_seasons.csv


GNN node order: ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']


09:09:42 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
09:09:42 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00
09:09:42 | src.data_loader | INFO | Train: 43824 rows (2019-01-01 00:00:00 → 2023-12-31 23:00:00)
09:09:42 | src.data_loader | INFO | Test : 8784 rows (2024-01-01 00:00:00 → 2024-12-31 23:00:00)
09:09:42 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -24 h
09:09:42 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
09:09:42 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
09:09:42 | src.models.gnn_stacking | INFO | build_gnn_sequence_dataset | seq_len=48 | weather_mode=perfect_forecast_

GNN Train: (29821, 48, 7, 35) | Test: (6349, 48, 7, 35)


09:09:43 | src.models.gnn_stacking | INFO | Training GNN-LSTM on cpu
09:09:46 | src.models.gnn_stacking | INFO | GNN-LSTM | seq_len=48  n_nodes=7  n_node_features=35  n_horizons=24  device=cpu
09:09:46 | src.models.gnn_stacking | INFO | Params: epochs=100  batch=64  patience=15  lr=0.00100
09:09:46 | src.models.gnn_stacking | INFO | Training samples=26839  val samples=2982
09:09:46 | src.models.gnn_stacking | INFO | -----------------------------------------------------------------
09:12:47 | src.models.gnn_stacking | INFO | Epoch   1/100  train=0.22385  val=0.09970  best=0.09970  lr=1.00e-03 *
09:15:54 | src.models.gnn_stacking | INFO | Epoch   2/100  train=0.09334  val=0.08056  best=0.08056  lr=1.00e-03 *
09:18:57 | src.models.gnn_stacking | INFO | Epoch   3/100  train=0.06864  val=0.07154  best=0.07154  lr=1.00e-03 *
09:21:56 | src.models.gnn_stacking | INFO | Epoch   4/100  train=0.05933  val=0.09108  best=0.07154  lr=1.00e-03  (no improve 1/15)
09:24:52 | src.models.gnn_stacking | 

Shared GNN predictions | train (29821, 7, 24) | test (6349, 7, 24)
Phase A complete in 72.1min.


In [4]:
# ════════════════════════════════════════════════════════════════════
# PHASE B — per-station GNN read-out + tree-stacking ensemble
# ════════════════════════════════════════════════════════════════════
n = len(STATIONS_TO_RUN)
b_start = time.time()

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'C3_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — stacking ...')
    t_station = time.time()
    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # The shared GNN is fixed; only the tabular/stacking pipeline is station-specific.
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station

    # Read out this station's node from the shared GNN's multi-output predictions.
    k = ALL_NODES.index(station)
    gnn_train_k = gnn_train_preds[:, k, :]   # (n_gnn_train, 24)
    gnn_test_k  = gnn_test_preds[:, k, :]    # (n_gnn_test, 24)
    y_gnn_test_k = y_gnn_test_true[:, k, :]  # (n_gnn_test, 24)

    # ── GNN evaluation (this station) ─────────────────────────────────
    gnn_rows = []
    for h_idx, h in enumerate(HORIZONS):
        m = compute_metrics(y_gnn_test_k[:, h_idx], gnn_test_k[:, h_idx])
        gnn_rows.append({'Model': 'C3_GNN_LSTM_pfxf', 'Station': station, 'Horizon': h, **m})
    gnn_df = pd.DataFrame(gnn_rows)
    gnn_df.to_csv(paths['results'] / 'C3_GNN_metrics.csv', index=False)

    # ── Stacking ensemble ─────────────────────────────────────────────
    stack_models = {}
    stack_rows = []
    base_model_classes = make_base_models()

    for h in tqdm(HORIZONS, desc=f'Stacking [{station}]'):
        X_tr_h, y_tr_h = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        h_idx = h - 1

        # Align GNN OOF predictions length with tabular training rows
        n_gnn = len(gnn_train_k)
        n_tab = len(X_tr_h)
        if n_gnn <= n_tab:
            gnn_oof_h = np.full(n_tab, np.nan)
            gnn_oof_h[-n_gnn:] = gnn_train_k[:, h_idx]
            valid = ~np.isnan(gnn_oof_h)
            X_h_valid = X_tr_h[valid]
            y_h_valid = y_tr_h[valid]
            gnn_oof_valid = gnn_oof_h[valid]
        else:
            gnn_oof_valid = gnn_train_k[-n_tab:, h_idx]
            X_h_valid = X_tr_h
            y_h_valid = y_tr_h

        meta, _ = train_stacking_ensemble(
            X_h_valid, y_h_valid,
            gnn_oof_preds=gnn_oof_valid,
            save_path=paths['models'] / f'meta_learner_pfx_h{h}.pkl'
        )
        stack_models[h] = meta

    # ── Stacking test evaluation ──────────────────────────────────────
    meta_feat_cols = None  # determined per-horizon below

    for h in HORIZONS:
        X_tr_h, y_tr_h = fe.build_feature_matrix(train_df, horizon=h, weather_mode=WEATHER_MODE)
        X_te_h, y_te_h = fe.build_feature_matrix(test_df,  horizon=h, weather_mode=WEATHER_MODE)

        if meta_feat_cols is None:
            meta_feat_cols = [c for c in META_FEAT_COLS_BASE if c in X_te_h.columns]

        base_preds_test = []
        for name, factory in base_model_classes.items():
            m = factory()
            m.fit(X_tr_h.values, y_tr_h.values)
            base_preds_test.append(m.predict(X_te_h.values))

        h_idx = h - 1
        n_gnn_te = len(gnn_test_k)
        n_tab_te = len(X_te_h)
        gnn_h = np.zeros(n_tab_te)
        if n_gnn_te >= n_tab_te:
            gnn_h = gnn_test_k[-n_tab_te:, h_idx]
        else:
            gnn_h[-n_gnn_te:] = gnn_test_k[:, h_idx]

        base_preds_test.append(gnn_h)
        base_arr  = np.column_stack(base_preds_test)
        meta_feats = X_te_h[meta_feat_cols].values if meta_feat_cols else np.zeros((n_tab_te, 1))

        final_preds = predict_stacking(stack_models[h], base_arr, meta_feats)
        m = compute_metrics(y_te_h.values, final_preds)
        stack_rows.append({'Model': 'C3_Stacking_pfxf', 'Station': station, 'Horizon': h, **m})

    stack_df = pd.DataFrame(stack_rows)
    stack_df.to_csv(paths['results'] / 'C3_Stacking_metrics.csv', index=False)

    # Combined C3 metrics (GNN + Stacking) — this file acts as the checkpoint
    combined = pd.concat([gnn_df, stack_df], ignore_index=True)
    combined.to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - b_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore default TARGET module state
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')


[1/7] MzWarChrosci — stacking ...


Stacking [MzWarChrosci]:   0%|          | 0/24 [00:00<?, ?it/s]

10:21:46 | src.feature_engineering | INFO | build_feature_matrix | horizon=1 | weather_mode=perfect_forecast_full
10:21:46 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -1 h
10:21:46 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
10:21:46 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
10:21:46 | src.feature_engineering | INFO |   X: (38271, 45) | y mean=17.38, y std=12.69
10:21:46 | src.models.gnn_stacking | INFO | OOF [1/4]: fitting base model 'xgb' across 5 folds
10:21:47 | src.models.gnn_stacking | INFO |   fold 1/5 — train=4971 val=4970
10:21:48 | src.models.gnn_stacking | INFO |   fold 2/5 — train=9941 val=4970
10:21:48 | src.models.gnn_stacking | INFO |   fold 3/5 — train=14911 val=4970
10:21:49 | src.models.gnn_stacking | INFO |   fold 4/5 — train=19881 val=4970
10:21:50 | src.models.gnn_stacking | INFO |   fold 5/5 — train=24851 va

[1/7] MzWarChrosci — done | elapsed 9.7min | est. remaining 58.1min

[2/7] MzOtwoBrzozo — stacking ...


Stacking [MzOtwoBrzozo]:   0%|          | 0/24 [00:00<?, ?it/s]

...

11:11:37 | src.feature_engineering | INFO | build_feature_matrix | horizon=1 | weather_mode=perfect_forecast_full
11:11:37 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -1 h
11:11:37 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
11:11:37 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
11:11:37 | src.feature_engineering | INFO |   X: (38067, 46) | y mean=17.97, y std=14.82
11:11:37 | src.models.gnn_stacking | INFO | OOF [1/4]: fitting base model 'xgb' across 5 folds
11:11:37 | src.models.gnn_stacking | INFO |   fold 1/5 — train=4971 val=4970
11:11:38 | src.models.gnn_stacking | INFO |   fold 2/5 — train=9941 val=4970
11:11:39 | src.models.gnn_stacking | INFO |   fold 3/5 — train=14911 val=4970
11:11:40 | src.models.gnn_stacking | INFO |   fold 4/5 — train=19881 val=4970
11:11:41 | src.models.gnn_stacking | INFO |   fold 5/5 — train=24851 va

[6/7] MzPiasPulask — done | elapsed 10.6min | est. remaining 10.1min

[7/7] MzWarBajkowa — stacking ...


Stacking [MzWarBajkowa]:   0%|          | 0/24 [00:00<?, ?it/s]

11:22:11 | src.feature_engineering | INFO | build_feature_matrix | horizon=1 | weather_mode=perfect_forecast_full
11:22:11 | src.feature_engineering | INFO | perfect_forecast_full: shifted 8 MET + 10 HYSPLIT_NUM + 2 HYSPLIT_CAT by -1 h
11:22:11 | src.feature_engineering | INFO | One-hot encoded HYSPLIT categoricals: ['dir_48', 'dir_24'] → 8 dummy cols
11:22:11 | src.feature_engineering | INFO | Retaining 10 HYSPLIT numeric columns
11:22:11 | src.feature_engineering | INFO |   X: (38496, 46) | y mean=18.12, y std=15.63
11:22:11 | src.models.gnn_stacking | INFO | OOF [1/4]: fitting base model 'xgb' across 5 folds
11:22:12 | src.models.gnn_stacking | INFO |   fold 1/5 — train=4971 val=4970
11:22:13 | src.models.gnn_stacking | INFO |   fold 2/5 — train=9941 val=4970
11:22:14 | src.models.gnn_stacking | INFO |   fold 3/5 — train=14911 val=4970
11:22:15 | src.models.gnn_stacking | INFO |   fold 4/5 — train=19881 val=4970
11:22:16 | src.models.gnn_stacking | INFO |   fold 5/5 — train=24851 va

[7/7] MzWarBajkowa — done | elapsed 12.6min | est. remaining 0.0min

All stations complete in 145.1min total.
